In [ ]:
# Cell 1: 导入模块
import torch
from importlib import reload

# 重新加载所有模块
import spatial_encoder, temporal_encoder, cross_attention, sparse_representation
reload(spatial_encoder)
reload(temporal_encoder)
reload(cross_attention)
reload(sparse_representation)

from spatial_encoder import SpatialEncoder
from temporal_encoder import TemporalEncoder
from cross_attention import CrossAttention
from sparse_representation import SparseRepresentation

## 
# from IPython.core.interactiveshell import InteractiveShell

# # 1. 设置 torch 打印选项
# torch.set_printoptions(
#     threshold=10000,      # 超过这个元素数才截断（设大点）
#     linewidth=200,        # 每行最大字符数
#     precision=4,          # 小数位数
#     edgeitems=50,         # 每维显示的头尾元素数
#     formatter=None        # 不使用自定义格式化
# )

# # 2. 设置 IPython 输出显示上限（设大点）
# InteractiveShell.instance().display_limit = 2000
##

print("✓ 模块导入成功")

✓ 模块导入成功


In [19]:
# Cell 2: 测试 SpatialEncoder (AAM Module)
print("\n[测试 1] SpatialEncoder - AAM 模块")
print("=" * 50)

encoder = SpatialEncoder(
    input_dim=4,       # 论文：4 个特征
    embed_dim=64,      # 论文最优
    num_heads=1,       # 论文最优
    num_layers=1,      # 论文最优
    dropout_rate=0.5   # 论文最优
)

# STCA 实际空间输入：(batch, num_satellites=20, features=4)
x_spatial = torch.randn(32, 20, 4)  # 32 个样本，20 颗卫星
out_spatial = encoder(x_spatial)
print(f"空间输入 (卫星维度): {x_spatial.shape} → 输出：{out_spatial.shape}")
assert out_spatial.shape == (32, 20, 64), "空间输出应为 (batch, num_sats, embed_dim)"

print("✓ SpatialEncoder 测试通过")


[测试 1] SpatialEncoder - AAM 模块
空间输入 (卫星维度): torch.Size([32, 20, 4]) → 输出：torch.Size([32, 20, 64])
✓ SpatialEncoder 测试通过


In [20]:
# Cell 3: 测试 TemporalEncoder - STCA 实际输入
print("\n[测试 2] TemporalEncoder - LSTM-TFE 模块")
print("=" * 50)

t_encoder = TemporalEncoder(
    input_dim=4,         
    embed_dim=64,        
    num_layers=1,        
    dropout_rate=0.5,    
    bidirectional=False  
)

# STCA 实际时间输入：(batch, window_size=10, features=4)
x_temporal = torch.randn(32, 10, 4)  # 32 个样本，10 个时间步
out_temporal = t_encoder(x_temporal)
print(f"时间输入 (窗口维度): {x_temporal.shape} → 输出：{out_temporal.shape}")
assert out_temporal.shape == (32, 64), "时间输出应为 (batch, embed_dim)"

print("✓ TemporalEncoder 测试通过")


[测试 2] TemporalEncoder - LSTM-TFE 模块
时间输入 (窗口维度): torch.Size([32, 10, 4]) → 输出：torch.Size([32, 64])
✓ TemporalEncoder 测试通过


In [23]:
# Cell 4: 测试 CrossAttention (时空交叉注意力模块)
print("\n[测试 3] CrossAttention - 交叉注意力模块")
print("=" * 50)

cross_attn = CrossAttention(
    embed_dim=64,       # 论文最优
    num_heads=1,        # 论文最优
    dropout_rate=0.5    # 论文最优
)

# 论文公式 (17): Query 来自时间特征，Key/Value 来自空间特征
query = torch.randn(32, 1, 64)     # 时间特征 (batch, 1, embed_dim)
key = torch.randn(32, 20, 64)      # 空间特征 (batch, num_sats, embed_dim)
value = torch.randn(32, 20, 64)    # 空间特征

output = cross_attn(query, key, value)
print(f"Query: {query.shape}, Key: {key.shape}, Value: {value.shape}")
print(f"输出：{output.shape}")
assert output.shape == (32, 1, 64), "输出应为 (batch, 1, embed_dim)"

# 测试返回注意力权重
output, attn_weights = cross_attn(query, key, value, return_attention_weights=True)
print(f"注意力权重形状：{attn_weights.shape}")
assert attn_weights.shape == (32, 1, 20), "注意力权重应为 (batch, 1, num_sats)"

print("✓ CrossAttention 测试通过")


[测试 3] CrossAttention - 交叉注意力模块
Query: torch.Size([32, 1, 64]), Key: torch.Size([32, 20, 64]), Value: torch.Size([32, 20, 64])
输出：torch.Size([32, 1, 64])
注意力权重形状：torch.Size([32, 1, 20])
✓ CrossAttention 测试通过


In [24]:
# Cell 5: 测试 SparseRepresentation (稀疏表示模块)
print("\n[测试 4] SparseRepresentation - 稀疏表示模块")
print("=" * 50)

sparse = SparseRepresentation(embed_dim=64)

# 测试 3D 输入：使用 CrossAttention 时，输出 (B, 1, 64) → Sparse 压缩为 (B, 64)
z_3d = torch.randn(32, 1, 64)
out_3d = sparse(z_3d)
print(f"3D 输入 (来自 CrossAttention): {z_3d.shape} → 输出：{out_3d.shape}")
assert out_3d.shape == (32, 64), "3D 输入会被 squeeze 为 2D"

# 测试稀疏性：小值输入应该被压成 0
z_small = torch.randn(8, 64) * 0.1
out_small = sparse(z_small)
zero_ratio = (out_small.abs() < 1e-4).float().mean().item()
print(f"小值输入零值比例：{zero_ratio:.2%}")

print("✓ SparseRepresentation 测试通过")


[测试 4] SparseRepresentation - 稀疏表示模块
3D 输入 (来自 CrossAttention): torch.Size([32, 1, 64]) → 输出：torch.Size([32, 64])
小值输入零值比例：100.00%
✓ SparseRepresentation 测试通过
